In this project, we'll work with a dataset of submissions to popular technology site [Hacker News](https://news.ycombinator.com/).

![hacker news logo](https://s3.amazonaws.com/dq-content/354/hacker_news.jpg)

Hacker News is a site started by the startup incubator [Y Combinator](https://www.ycombinator.com/), where user-submitted stories (known as "posts") receive votes and comments, similar to reddit. Hacker News is extremely popular in technology and startup circles, and posts that make it to the top of the Hacker News listings can get hundreds of thousands of visitors as a result.

You can find the data set [here](https://www.kaggle.com/hacker-news/hacker-news-posts), but note that we have reduced from almost 300,000 rows to approximately 20,000 rows by removing all submissions that didn't receive any comments and then randomly sampling from the remaining submissions. You can download this downsampled data [here](https://dq-content.s3.amazonaws.com/356/hacker_news.csv) or from the jupyter notebook workspace by clicking **File** -> **Open** -> **hacker\_news.csv** -> **File** -> **Download**.

Below are descriptions of the columns:

- `id`: the unique identifier from Hacker News for the post
- `title`: the title of the post
- `url`: the URL that the posts links to, if the post has a URL
- `num_points`: the number of points the post acquired, calculated as the total number of upvotes minus the total number of downvotes
- `num_comments`: the number of comments on the post
- `author`: the username of the person who submitted the post
- `created_at`: the date and time of the post's submission

### Loading & Exploring Dataset

In [1]:
import pandas as pd
import datetime as dt

hacker = pd.read_csv(r"C:\Users\ntxhi\Downloads\hacker_news.csv", parse_dates=['created_at'])

In [2]:
hacker.head()

,id,title,url,num_points,num_comments,author,created_at
0,12224879,Interactive Dynamic Video,http://www.interactivedynamicvideo.com/,386,52,ne0phyte,2016-08-04 11:52:00
1,10975351,How to Use Open Source and Shut the Fuck Up at...,http://hueniverse.com/2016/01/26/how-to-use-op...,39,10,josep2,2016-01-26 19:30:00
2,11964716,Florida DJs May Face Felony for April Fools' W...,http://www.thewire.com/entertainment/2013/04/f...,2,1,vezycash,2016-06-23 22:20:00
3,11919867,Technology ventures: From Idea to Enterprise,https://www.amazon.com/Technology-Ventures-Ent...,3,1,hswarna,2016-06-17 00:01:00
4,10301696,Note by Note: The Making of Steinway L1037 (2007),http://www.nytimes.com/2007/11/07/movies/07ste...,8,2,walterbell,2015-09-30 04:12:00


In [3]:
hacker.describe()

,id,num_points,num_comments,created_at
count,2.010000e+04,20100.000000,20100.000000,20100
mean,1.131753e+07,50.296070,24.802289,2016-03-15 02:42:35.991044864
min,1.017691e+07,1.000000,1.000000,2015-09-06 05:56:00
25%,1.070176e+07,3.000000,1.000000,2015-12-09 03:41:00
50%,1.128445e+07,9.000000,3.000000,2016-03-14 18:15:00
75%,1.192607e+07,54.000000,21.000000,2016-06-17 23:03:45
max,1.257898e+07,2553.000000,1733.000000,2016-09-26 03:13:00
std,6.964399e+05,107.107687,56.107340,NaN


In [4]:
hacker.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20100 entries, 0 to 20099
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   id            20100 non-null  int64         
 1   title         20100 non-null  object        
 2   url           17660 non-null  object        
 3   num_points    20100 non-null  int64         
 4   num_comments  20100 non-null  int64         
 5   author        20100 non-null  object        
 6   created_at    20100 non-null  datetime64[ns]
dtypes: datetime64[ns](1), int64(3), object(3)
memory usage: 1.1+ MB


We're specifically interested in posts with titles that begin with either `Ask HN` or `Show HN`. Users submit `Ask HN` posts to ask the Hacker News community a specific question. Below are a few examples:

```
Ask HN: How to improve my personal website?
Ask HN: Am I the only one outraged by Twitter shutting down share counts?
Ask HN: Aby recent changes to CSS that broke mobile?
```

Likewise, users submit `Show HN` posts to show the Hacker News community a project, product, or just something interesting. Below are a few examples:

```
Show HN: Wio Link  ESP8266 Based Web of Things Hardware Development Platform'
Show HN: Something pointless I made
Show HN: Shanhu.io, a programming playground powered by e8vm
```

### Do `Ask HN` or `Show HN` receive more comments on average?

In [5]:
# Number of posts with titles that begin with Ask HN

ask_posts = hacker.loc[hacker['title'].str.lower().str.startswith('ask hn')]
ask_posts['title'].count()

np.int64(1744)

In [6]:
# Average comments of posts with titles that begin with Ask HN

ask_posts['num_comments'].mean()

np.float64(14.038417431192661)

In [7]:
# Number of posts with titles that begin with Show HN

show_posts = hacker.loc[hacker['title'].str.lower().str.startswith('show hn')]
show_posts['title'].count()

np.int64(1162)

In [8]:
# Average comments of posts with titles that begin with Show HN

show_posts['num_comments'].mean()


np.float64(10.31669535283993)

In [9]:
# Number of other posts
len(hacker) - ask_posts.count() - show_posts.count()


id              17194
title           17194
url             18971
num_points      17194
num_comments    17194
author          17194
created_at      17194
dtype: int64

`Ask HN` receive more comments.

And on average, ask posts receive more comments than show posts.


### Do posts created at a certain time receive more comments on average?

We've determined that, on average, ask posts receive more comments than show posts. Since ask posts are more likely to receive comments, we'll focus our remaining analysis just on these posts.

Next, we'll determine if ask posts created at a certain time are more likely to attract comments.

In [10]:
hacker['time_created'] = hacker['created_at'].dt.hour.astype(str) + ':00'

We can clearly see that ask posts are mostly created after afternoon (14h-19h) and evening (21h). We can guess people have more free time outside office hour to create new post.

In [11]:
ask_posts = hacker.loc[hacker['title'].str.lower().str.startswith('ask hn')]

In [12]:
ask_posts.pivot_table(index='time_created', 
                      values= 'num_comments',
                      aggfunc='mean') \
         .sort_values('num_comments', ascending=False) \
         .head()

,num_comments
time_created,
15:00,38.594828
2:00,23.810345
20:00,21.525000
16:00,16.796296
21:00,16.009174


### Conclusion

Our analysis shows the hours in which most comments are posted on average. If we refer to the [dataset documentation](https://www.kaggle.com/hacker-news/hacker-news-posts) we see that the times refer to US Eastern Time. For me here in Vietnam, if I want to maximise my chances of getting a high number of comments on a post in the `Ask HN` category, I should post at 8am, 9am and 10pm as GMT is 12 hours ahead. Of note, eventhough posts created at `15:00` gets the most og interaction, it's nearly impossible for me to create a new post in this time (as it's 3am), the same happens with `16:00` (i.e. 4am in Vietnam).